<a href="https://colab.research.google.com/github/iLevyTate/ITASORL/blob/ColabFullRun/notebooks/colab_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ITASORL: full end-to-end (Colab or local Jupyter)

Runs **pytest + all experiments** via `scripts/run_e2e.py`, records everything under
`fullruns/<MMDDYYYY>/` (or Google Drive on Colab), and saves **`bundle.zip`** for offline review.

**Live output while running:** `combined.log` and `status.json` update incrementally.
In a second terminal (local): `python scripts/watch_run.py --follow`

**Open on Google Colab (GPU):**
[colab_gpu.ipynb on GitHub](https://colab.research.google.com/github/iLevyTate/ITASORL/blob/main/notebooks/colab_gpu.ipynb)

**Before you start (Colab):** Runtime, then Change runtime type, then pick **your hardware** (GPU recommended; any CUDA device works: T4, L4, A100, etc.). This notebook does not request a specific GPU. Mount Drive when prompted.

**Colab timeouts (read this):** Free Colab can disconnect after ~90 min on GPU or if the tab sleeps. Runs use **local disk** (`fullruns/`) and mirror logs to Drive after each step. If the run stops, set `RESUME_RUN_DIR` to the mirrored Drive folder and re-run.

**Local Jupyter / VS Code:** open this file from the repo, set `MOUNT_DRIVE = False`, run all cells from repo root context (clone cell sets `REPO_DIR`).

**After the run, read:** `SUMMARY.md` in the run folder (plain English verdict).

In [1]:
# Config
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/iLevyTate/ITASORL.git"
BRANCH = "main"

# Colab clones to /content; local Jupyter: point at your checkout
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    REPO_DIR = "/content/ITASORL"
except ImportError:
    IN_COLAB = False
    REPO_DIR = str(Path.cwd().resolve())
    # If you opened the notebook from notebooks/, go up one level
    if not (Path(REPO_DIR) / "scripts" / "run_e2e.py").is_file():
        REPO_DIR = str(Path(REPO_DIR).parent)

RUN_MODE = "quick"          # "quick" (~30-60 min) | "full" (hours; may exceed Colab limits)
SKIP_STEPS = []             # e.g. ["expB2"] to skip longest step

MOUNT_DRIVE = IN_COLAB      # Colab: mirror live logs + save final bundle to Drive
DRIVE_RESULTS = "/content/drive/MyDrive/ITASORL_results"
SAVE_TO_DRIVE = IN_COLAB    # copy full run folder + bundle.zip to Drive when done
DOWNLOAD_BUNDLE = IN_COLAB  # browser download (extra fallback on Colab)

# Colab crash recovery: run on fast local disk; mirror to Drive after each step.
# Do NOT set RESULTS_ON_DRIVE = True (Drive FUSE is too slow and often fails mid-run).
#
# FRESH RUN (default): leave FRESH_RUN = True and RESUME_RUN_DIR = None
FRESH_RUN = True
RESULTS_ON_DRIVE = False
RUN_ID = datetime.now().strftime("%m%d%Y_%H%M%S")
RESUME_RUN_DIR = None
COPY_DRIVE_RESUME_TO_LOCAL = True  # when resuming from Drive, copy to /content first

# ---------------------------------------------------------------------------
# RESUME RUN (after Colab timeout): uncomment block below, comment FRESH_RUN
# 1. Mount Drive, find MyDrive/ITASORL_results/<your_run_folder>
# 2. Open manifest.json there; steps with "status": "ok" are skipped
# 3. Uncomment, paste your folder path, set FRESH_RUN = False
# 4. Re-run cells from config through keep-alive + run (not full notebook)
# ---------------------------------------------------------------------------
# FRESH_RUN = False
# RESUME_RUN_DIR = "/content/drive/MyDrive/ITASORL_results/MMDDYYYY_HHMMSS"
# RESULTS_ON_DRIVE = False
# RUN_ID = None
# COPY_DRIVE_RESUME_TO_LOCAL = True

In [2]:
import os, subprocess, sys, shutil, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=check)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
elif not IN_COLAB:
    print(f"Local mode (repo: {REPO_DIR})")

Mounted at /content/drive


In [3]:
if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        sh(f"cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
    else:
        sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}")
else:
    print("Local mode: using existing checkout (skip clone)")
    if not (Path(REPO_DIR) / "itasorl").is_dir():
        raise FileNotFoundError(f"Expected itasorl/ under {REPO_DIR}; set REPO_DIR in config cell")

os.chdir(REPO_DIR)
sh("git log -1 --oneline")
print("REPO_DIR:", REPO_DIR)

$ git clone --branch main --single-branch https://github.com/iLevyTate/ITASORL.git /content/ITASORL
$ git log -1 --oneline
REPO_DIR: /content/ITASORL


In [4]:
sh(f"{sys.executable} -m pip install -q --upgrade pip")
dev = os.path.join(REPO_DIR, "requirements-dev.txt")
if os.path.isfile(dev):
    sh(f"{sys.executable} -m pip install -q -r requirements-dev.txt")
else:
    sh(f"{sys.executable} -m pip install -q -r requirements.txt pytest")

$ /usr/bin/python3 -m pip install -q --upgrade pip
$ /usr/bin/python3 -m pip install -q -r requirements-dev.txt


In [ ]:
import torch

# Hardware is your choice: Runtime > Change runtime type (not hardcoded to T4).
if torch.cuda.is_available():
    print("CUDA: True")
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("CUDA: False")
    print("No GPU detected. Runtime > Change runtime type > pick a GPU, then re-run this cell.")
    print("CPU-only works but expB2 will be much slower.")

In [ ]:
# Keep-alive (Colab only): run immediately before the long run cell.
if IN_COLAB:
    from IPython.display import display, Javascript

    display(Javascript("""
    (function () {
      const intervalMs = 60000;
      setInterval(() => {
        fetch('/gen_' + Date.now()).catch(() => {});
      }, intervalMs);
      console.log('ITASORL keep-alive active every', intervalMs / 1000, 's');
    })();
    """))
    print("Keep-alive enabled. Leave this tab open until the run finishes.")
else:
    print("Local mode: keep-alive not needed.")

In [ ]:
cmd = [sys.executable, "scripts/run_e2e.py"]
if RUN_MODE == "quick":
    cmd.append("--quick")
elif RUN_MODE != "full":
    raise ValueError("RUN_MODE must be 'quick' or 'full'")
for step in SKIP_STEPS:
    cmd.extend(["--skip", step])

run_dir = None
if RESUME_RUN_DIR:
    src = Path(RESUME_RUN_DIR)
    if not src.is_dir():
        raise FileNotFoundError(f"RESUME_RUN_DIR not found: {src}")
    if COPY_DRIVE_RESUME_TO_LOCAL and str(src).startswith("/content/drive"):
        run_dir = Path(REPO_DIR) / "fullruns" / "_resume_local" / src.name
        if not (run_dir / "manifest.json").is_file():
            print("Copying Drive run to local disk for resume:", run_dir, flush=True)
            shutil.copytree(src, run_dir, dirs_exist_ok=True)
        else:
            print("Using cached local copy for resume:", run_dir, flush=True)
    else:
        run_dir = src
    cmd.extend(["--resume", str(run_dir)])
    print("Resuming run:", run_dir, flush=True)
elif RESULTS_ON_DRIVE and MOUNT_DRIVE:
    run_dir = Path(DRIVE_RESULTS) / RUN_ID
    run_dir.mkdir(parents=True, exist_ok=True)
    cmd.extend(["--results-dir", str(run_dir)])
    print("New run; results on Drive:", run_dir, flush=True)
else:
    print("New run; results on local disk under fullruns/ (mirrored to Drive if mounted)", flush=True)

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"
if MOUNT_DRIVE:
    run_env["ITASORL_DRIVE_SYNC"] = DRIVE_RESULTS
    print("Live log mirror -> Drive:", DRIVE_RESULTS, flush=True)

def _print_failure_diagnostics():
    import json
    ptr = Path(REPO_DIR) / "results" / "LATEST_RUN.txt"
    rd = run_dir if run_dir is not None else (Path(ptr.read_text().strip()) if ptr.is_file() else None)
    if rd is None or not rd.is_dir():
        print("No run directory found for diagnostics.", flush=True)
        return
    print("\n" + "=" * 72, flush=True)
    print("RUN FAILED. Diagnostics for:", rd, flush=True)
    mf = rd / "manifest.json"
    if mf.is_file():
        steps = json.loads(mf.read_text()).get("steps", {})
        failed = [n for n, s in steps.items() if s.get("status") == "failed"]
        ok = [n for n, s in steps.items() if s.get("status") == "ok"]
        if ok:
            print("Completed steps:", ", ".join(ok), flush=True)
        if failed:
            print("Failed steps:", ", ".join(failed), flush=True)
            for name in failed:
                log = rd / "steps" / f"{name}.log"
                if log.is_file():
                    tail = log.read_text(errors="replace").splitlines()[-40:]
                    print(f"\n--- tail {name}.log ---", flush=True)
                    print("\n".join(tail), flush=True)
    cl = rd / "combined.log"
    if cl.is_file():
        tail = cl.read_text(errors="replace").splitlines()[-30:]
        print("\n--- tail combined.log ---", flush=True)
        print("\n".join(tail), flush=True)
    print("To resume: set RESUME_RUN_DIR to this folder (Drive or local) and re-run.", flush=True)
    print("=" * 72, flush=True)

print("If Colab disconnects: set RESUME_RUN_DIR to the mirrored Drive folder, then re-run keep-alive + this cell.", flush=True)

t0 = time.perf_counter()
proc = subprocess.run(cmd, cwd=REPO_DIR, env=run_env)
if proc.returncode != 0:
    _print_failure_diagnostics()
    raise subprocess.CalledProcessError(proc.returncode, proc.args)
print(f"Wall time: {(time.perf_counter() - t0) / 60:.1f} min")

In [ ]:
# RESUME-ONLY RUN (commented out for now)
# If Colab timed out: uncomment this entire cell and run it INSTEAD of the run cell above.
# Prerequisite: config cell must have FRESH_RUN = False and RESUME_RUN_DIR set.
#
# RESUME_FOLDER = "/content/drive/MyDrive/ITASORL_results/MMDDYYYY_HHMMSS"
#
# cmd = [sys.executable, "scripts/run_e2e.py", "--resume", RESUME_FOLDER]
# if RUN_MODE == "quick":
#     cmd.append("--quick")
# for step in SKIP_STEPS:
#     cmd.extend(["--skip", step])
#
# run_env = os.environ.copy()
# run_env["PYTHONUNBUFFERED"] = "1"
# if MOUNT_DRIVE:
#     run_env["ITASORL_DRIVE_SYNC"] = DRIVE_RESULTS
#
# manifest = Path(RESUME_FOLDER) / "manifest.json"
# if manifest.is_file():
#     import json
#     done = [k for k, v in json.loads(manifest.read_text()).get("steps", {}).items() if v.get("status") == "ok"]
#     print("Already completed:", ", ".join(done) or "(none)")
# print("Resuming:", RESUME_FOLDER, flush=True)
# t0 = time.perf_counter()
# subprocess.run(cmd, cwd=REPO_DIR, check=True, env=run_env)
# print(f"Wall time: {(time.perf_counter() - t0) / 60:.1f} min")

print("Resume cell idle (uncomment block above when needed).")

In [ ]:
latest_ptr = Path(REPO_DIR) / "results" / "LATEST_RUN.txt"
RUN_DIR = Path(latest_ptr.read_text().strip())
BUNDLE = RUN_DIR / "bundle.zip"
SUMMARY = RUN_DIR / "SUMMARY.md"

print("Run directory:", RUN_DIR)
print("Bundle:", BUNDLE, "exists:", BUNDLE.is_file())
print("\n" + "=" * 72)
print(SUMMARY.read_text())
print("=" * 72)

In [ ]:
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Missing bundle: {BUNDLE}")

print("Local fullruns run:", RUN_DIR)
print("Local bundle:", BUNDLE)

if SAVE_TO_DRIVE and MOUNT_DRIVE:
    drive_root = Path(DRIVE_RESULTS)
    drive_run = drive_root / RUN_DIR.name
    shutil.copytree(RUN_DIR, drive_run, dirs_exist_ok=True)
    print("Full run copied to Drive:", drive_run)

    drive_bundle = drive_root / f"{RUN_DIR.name}_bundle.zip"
    shutil.copy2(BUNDLE, drive_bundle)
    print("Bundle copied to Drive:", drive_bundle)

    shutil.copy2(SUMMARY, drive_root / f"{RUN_DIR.name}_SUMMARY.md")
    print("SUMMARY copied to Drive")

if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(BUNDLE))
    print("Browser download started for bundle.zip")
else:
    print("Open SUMMARY:", SUMMARY)

## What's in the bundle

| File | Purpose |
|------|---------|
| `SUMMARY.md` | Plain English outcome. **Read this first.** |
| `status.json` | Live step + last line (updated during run) |
| `manifest.json` | Step timings, status, artifact index |
| `combined.log` | Full stdout (updated live during run) |
| `steps/*.json` | Parsed metrics (AUROC per drift, etc.) |
| `artifacts/` | Figures + `expB2_results.json` |

**While running (local):** `python scripts/watch_run.py --follow`

**While running (Colab + Drive):** open `MyDrive/ITASORL_results/<run>/combined.log`

**If Colab timed out:** check Drive for your `RUN_ID` folder. Open `manifest.json` to see which steps finished (`status: ok`). Set `RESUME_RUN_DIR` to that folder in the config cell, then re-run from the keep-alive cell.

Unzip locally and open `SUMMARY.md` to decide whether the organism encoded world identity.